# Media & Telecom Analytics Dashboard
Interactive KPI summary for Subscriber Health, Content Performance, and Ad Revenue.

In [ ]:
# Load gold tables
df_scorecard  = spark.read.format('delta').table('gold_subscriber_scorecards').toPandas()
df_content    = spark.read.format('delta').table('gold_content_performance').toPandas()
df_churn      = spark.read.format('delta').table('gold_churn_analysis').toPandas()
df_ad         = spark.read.format('delta').table('gold_ad_revenue').toPandas()
df_weekly     = spark.read.format('delta').table('gold_weekly_viewing_trends').toPandas()
df_silver_cc  = spark.read.format('delta').table('silver_content_catalog').toPandas()[['content_id', 'title']]

In [ ]:
total_subs      = int(df_scorecard['total_subscribers'].sum())
active_subs     = int(df_scorecard['active_subscribers'].sum())
total_mrr       = round(df_scorecard['total_mrr'].sum(), 2)
avg_churn_rate  = round(df_scorecard['churned_count'].sum() / total_subs * 100, 1)
avg_arpu        = round(df_scorecard['total_mrr'].sum() / active_subs, 2)
total_ad_rev    = round(df_ad['total_revenue'].sum(), 2)

print(f"Total Subscribers  : {total_subs:,}")
print(f"Active Subscribers : {active_subs:,}")
print(f"Monthly MRR        : ${total_mrr:,.2f}")
print(f"Overall Churn Rate : {avg_churn_rate}%")
print(f"Average ARPU       : ${avg_arpu:,.2f}")
print(f"Total Ad Revenue   : ${total_ad_rev:,.2f}")

In [ ]:
# Top 10 content by views (with titles)
top_content = (
    df_content
    .merge(df_silver_cc, on='content_id', how='left')
    .sort_values('total_views', ascending=False)
    .head(10)[['title', 'genre', 'content_type', 'total_views', 'avg_completion_pct', 'avg_rating', 'unique_viewers']]
)

# Ad revenue by type
ad_by_type = df_ad.groupby('ad_type', as_index=False).agg(
    total_revenue=('total_revenue', 'sum'),
    total_impressions=('total_impressions', 'sum'),
    avg_ctr=('avg_ctr', 'mean'),
).sort_values('total_revenue', ascending=False)

html = f"""
<!DOCTYPE html><html><head>
<meta charset="utf-8">
<title>Media & Telecom Analytics Dashboard</title>
<style>
  body {{ font-family: Segoe UI, Arial, sans-serif; background:#f4f6f9; margin:0; padding:20px; }}
  h1   {{ color:#0078d4; border-bottom:3px solid #0078d4; padding-bottom:8px; }}
  h2   {{ color:#323130; margin-top:30px; }}
  .kpis {{ display:flex; gap:16px; flex-wrap:wrap; margin:20px 0; }}
  .kpi  {{ background:#fff; border-radius:8px; padding:20px 28px; box-shadow:0 2px 6px rgba(0,0,0,.08);
           min-width:160px; text-align:center; }}
  .kpi .val {{ font-size:2em; font-weight:700; color:#0078d4; }}
  .kpi .lbl {{ font-size:.85em; color:#605e5c; margin-top:4px; }}
  table {{ border-collapse:collapse; width:100%; background:#fff; border-radius:8px;
           box-shadow:0 2px 6px rgba(0,0,0,.08); overflow:hidden; margin-bottom:30px; }}
  th    {{ background:#0078d4; color:#fff; padding:10px 14px; text-align:left; font-size:.9em; }}
  td    {{ padding:9px 14px; border-bottom:1px solid #edebe9; font-size:.88em; }}
  tr:hover td {{ background:#f3f9ff; }}
  .badge {{ display:inline-block; padding:2px 10px; border-radius:12px; font-size:.8em; font-weight:600; }}
  .Healthy  {{ background:#dff6dd; color:#107c10; }}
  .Monitor  {{ background:#cceaff; color:#0078d4; }}
  .At.Risk  {{ background:#fff4ce; color:#b7610e; }}
  .Critical {{ background:#fde7e9; color:#a80000; }}
</style></head><body>
<h1>📺 Media & Telecom Analytics Dashboard</h1>

<div class="kpis">
  <div class="kpi"><div class="val">{total_subs:,}</div><div class="lbl">Total Subscribers</div></div>
  <div class="kpi"><div class="val">{active_subs:,}</div><div class="lbl">Active Subscribers</div></div>
  <div class="kpi"><div class="val">${total_mrr:,.0f}</div><div class="lbl">Monthly MRR</div></div>
  <div class="kpi"><div class="val">{avg_churn_rate}%</div><div class="lbl">Churn Rate</div></div>
  <div class="kpi"><div class="val">${avg_arpu}</div><div class="lbl">Avg ARPU</div></div>
  <div class="kpi"><div class="val">${total_ad_rev:,.2f}</div><div class="lbl">Ad Revenue</div></div>
</div>

<h2>Top 10 Content by Views</h2>
<table>
  <tr><th>Title</th><th>Genre</th><th>Type</th><th>Views</th><th>Unique Viewers</th><th>Avg Completion</th><th>Avg Rating</th></tr>
"""
for _, r in top_content.iterrows():
    rating = f"{r['avg_rating']:.1f}/5" if r['avg_rating'] == r['avg_rating'] else 'N/A'
    html += f"""
  <tr>
    <td>{r['title']}</td>
    <td>{r['genre']}</td>
    <td>{r['content_type']}</td>
    <td>{int(r['total_views']):,}</td>
    <td>{int(r['unique_viewers']):,}</td>
    <td>{r['avg_completion_pct']:.1f}%</td>
    <td>{rating}</td>
  </tr>"""

html += """
</table>

<h2>Ad Revenue by Type</h2>
<table>
  <tr><th>Ad Type</th><th>Total Revenue</th><th>Total Impressions</th><th>Avg CTR</th></tr>
"""
for _, r in ad_by_type.iterrows():
    html += f"""
  <tr>
    <td>{r['ad_type']}</td>
    <td>${r['total_revenue']:,.2f}</td>
    <td>{int(r['total_impressions']):,}</td>
    <td>{r['avg_ctr']:.2f}%</td>
  </tr>"""

html += """
</table>

<h2>Subscriber Health by Plan & Region</h2>
<table>
  <tr><th>Plan</th><th>Region</th><th>Total</th><th>Active</th><th>Churn Rate</th><th>ARPU</th><th>MRR</th><th>Health</th></tr>
"""
for _, r in df_scorecard.sort_values('churn_rate', ascending=False).iterrows():
    band = str(r['health_band']).replace(' ', '.')
    html += f"""
  <tr>
    <td>{r['plan_type']}</td>
    <td>{r['region']}</td>
    <td>{int(r['total_subscribers']):,}</td>
    <td>{int(r['active_subscribers']):,}</td>
    <td>{r['churn_rate']}%</td>
    <td>${r['arpu']}</td>
    <td>${r['total_mrr']:,.2f}</td>
    <td><span class="badge {band}">{r['health_band']}</span></td>
  </tr>"""

html += """
</table>
</body></html>"""

displayHTML(html)

In [ ]:
# Save dashboard as HTML artifact
with open('/lakehouse/default/Files/media_dashboard.html', 'w') as f:
    f.write(html)
print('Dashboard saved to lakehouse Files/media_dashboard.html')